# Step 14: Final Model Audit

Audit of the existing models to determine the best overall forecasting method for each crop.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
BASE_DIR = Path('..').resolve()
if str(BASE_DIR) not in sys.path: sys.path.insert(0, str(BASE_DIR))

from src.forecast_engine import load_forecast_config, build_forecast_config
config = build_forecast_config(save=True)


In [2]:
CROPS = ['rice', 'tomato', 'wheat', 'cotton']
records = []
for crop in CROPS:
    meta_path = BASE_DIR / 'models' / (f'model_metadata.json' if crop == 'rice' else f'{crop}_model_metadata.json')
    with open(meta_path) as f: meta = json.load(f)
    rec = {
        'Crop': crop.capitalize(),
        'Best ML Model': meta.get('model_name', 'Unknown'),
        'ML MAE': meta.get('MAE'),
        'ML RMSE': meta.get('RMSE'),
        'ML MAPE': meta.get('MAPE'),
        'ML R2': meta.get('R2'),
        'Persistence MAE': meta.get('persistence_baseline_MAE'),
        'Persistence RMSE': meta.get('persistence_baseline_RMSE'),
        'Persistence MAPE': meta.get('persistence_baseline_MAPE'),
        'ML Improvement vs Persistence': meta.get('improvement_vs_persistence_pct'),
        'Best ML Method': meta.get('model_name', 'Unknown'),
        'Best Overall Method': config[crop]['method']
    }
    records.append(rec)
df = pd.DataFrame(records)
(BASE_DIR / 'outputs' / 'final').mkdir(parents=True, exist_ok=True)
df.to_csv(BASE_DIR / 'outputs' / 'final' / 'crop_model_summary.csv', index=False)
display(df)
for crop in CROPS:
    print(f"{crop.capitalize()}: {config[crop]['reason']}")


,Crop,Best ML Model,ML MAE,ML RMSE,ML MAPE,ML R2,Persistence MAE,Persistence RMSE,Persistence MAPE,ML Improvement vs Persistence,Best ML Method,Best Overall Method
0,Rice,Gradient Boosting,232.1300,396.4600,5.0500,0.7236,120.9200,NaN,NaN,-91.9700,Gradient Boosting,persistence
1,Tomato,Random Forest,434.3620,549.8029,19.0385,0.6692,409.5864,604.7440,16.0279,-6.0489,Random Forest,persistence
2,Wheat,Gradient Boosting,68.2390,107.6109,2.2389,0.9505,66.9079,127.1733,2.2432,-1.9895,Gradient Boosting,persistence
3,Cotton,Gradient Boosting,212.2251,293.4315,2.5700,-0.9184,103.6250,164.4023,1.2717,-104.8011,Gradient Boosting,persistence


Rice: Persistence MAE ₹120.92 is <= ML MAE ₹232.13 (improvement -92.0%); persistence preferred
Tomato: Persistence MAE ₹409.59 is <= ML MAE ₹434.36 (improvement -6.0%); persistence preferred
Wheat: Persistence MAE ₹66.91 is <= ML MAE ₹68.24 (improvement -2.0%); persistence preferred
Cotton: Persistence MAE ₹103.62 is <= ML MAE ₹212.23 (improvement -104.8%); persistence preferred
